# LUAD Survival Analysis — RNA-seq Processing
**Author:** Parth Shringarpure  
**Date:** June 2026  
**Goal:** Load TCGA-LUAD RNA-seq gene expression data, match patients to clinical survival labels, 
normalise expression values, select top variable genes, and save a clean expression matrix 
ready for DeepSurv training.

**Input:** `data/raw/.../LUAD.rnaseqv2...RSEM_genes_normalized__data.data.txt`  
**Output:** `data/processed/expression_matrix.csv`

## Step 1: Imports and Setup

In [3]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

os.chdir('/Users/parthshringarpure/Desktop/AI/Projects/luad_survival')

print("All imports successful")
print(f"Working directory: {os.getcwd()}")

All imports successful
Working directory: /Users/parthshringarpure/Desktop/AI/Projects/luad_survival


## Step 2: Peek at the Raw RNA-seq File

Before loading the full matrix, we inspect the first few lines to understand 
the file structure — how many rows, columns, what the headers look like, 
and whether it needs any special handling.

In [4]:
RNASEQ_PATH = (
    "data/raw/gdac.broadinstitute.org_LUAD.Merge_rnaseqv2__illuminahiseq_rnaseqv2"
    "__unc_edu__Level_3__RSEM_genes_normalized__data.Level_3.2016012800.0.0/"
    "LUAD.rnaseqv2__illuminahiseq_rnaseqv2__unc_edu__Level_3__RSEM_genes_normalized"
    "__data.data.txt"
)

# Peek at just the first 5 rows and 6 columns
# without loading the whole file
peek = pd.read_csv(RNASEQ_PATH, sep='\t', nrows=5, index_col=0)
print(f"Peek shape: {peek.shape}")
print(f"\nFirst few column names (patient IDs):")
for col in peek.columns[:5]:
    print(f"  {col}")
print(f"\nFirst few row names (genes):")
for idx in peek.index[:5]:
    print(f"  {idx}")
print(f"\nSample values:")
print(peek.iloc[:4, :4])

Peek shape: (5, 576)

First few column names (patient IDs):
  TCGA-05-4244-01A-01R-1107-07
  TCGA-05-4249-01A-01R-1107-07
  TCGA-05-4250-01A-01R-1107-07
  TCGA-05-4382-01A-01R-1206-07
  TCGA-05-4384-01A-01R-1755-07

First few row names (genes):
  gene_id
  ?|100130426
  ?|100133144
  ?|100134869
  ?|10357

Sample values:
                  TCGA-05-4244-01A-01R-1107-07 TCGA-05-4249-01A-01R-1107-07  \
Hybridization REF                                                             
gene_id                       normalized_count             normalized_count   
?|100130426                             0.0000                       0.0000   
?|100133144                            10.0113                       7.1957   
?|100134869                            11.2820                      12.4436   

                  TCGA-05-4250-01A-01R-1107-07 TCGA-05-4382-01A-01R-1206-07  
Hybridization REF                                                            
gene_id                       normalized_count

## Step 3: Load the Full RNA-seq Matrix

The file has two quirks we handle on load:
- **Row 1 is a junk header** ("normalized_count" repeated) — we skip it with `skiprows`
- **Patient IDs are long barcodes** — we trim to first 12 chars and lowercase to match clinical data
- **Gene names are in format `SYMBOL|entrez_id`** — we clean these after loading

In [5]:
# Load full matrix
# skiprows=[1] skips the second header row (normalized_count row)
print("Loading RNA-seq matrix — this may take 10-20 seconds...")

rna_raw = pd.read_csv(
    RNASEQ_PATH,
    sep='\t',
    index_col=0,
    skiprows=[1],
    low_memory=False
)

print(f"Raw shape: {rna_raw.shape}  (genes × patients)")
print(f"\nFirst 3 gene names: {rna_raw.index[:3].tolist()}")
print(f"First 3 patient IDs: {rna_raw.columns[:3].tolist()}")

Loading RNA-seq matrix — this may take 10-20 seconds...
Raw shape: (20531, 576)  (genes × patients)

First 3 gene names: ['?|100130426', '?|100133144', '?|100134869']
First 3 patient IDs: ['TCGA-05-4244-01A-01R-1107-07', 'TCGA-05-4249-01A-01R-1107-07', 'TCGA-05-4250-01A-01R-1107-07']


In [5]:
rna_raw.head(20)

,TCGA-05-4244-01A-01R-1107-07,TCGA-05-4249-01A-01R-1107-07,TCGA-05-4250-01A-01R-1107-07,TCGA-05-4382-01A-01R-1206-07,TCGA-05-4384-01A-01R-1755-07,TCGA-05-4389-01A-01R-1206-07,TCGA-05-4390-01A-02R-1755-07,TCGA-05-4395-01A-01R-1206-07,TCGA-05-4396-01A-21R-1858-07,TCGA-05-4397-01A-01R-1206-07,...,TCGA-NJ-A4YG-01A-22R-A262-07,TCGA-NJ-A4YI-01A-11R-A262-07,TCGA-NJ-A4YP-01A-11R-A262-07,TCGA-NJ-A4YQ-01A-11R-A262-07,TCGA-NJ-A55A-01A-11R-A262-07,TCGA-NJ-A55O-01A-11R-A262-07,TCGA-NJ-A55R-01A-11R-A262-07,TCGA-NJ-A7XG-01A-12R-A39D-07,TCGA-O1-A52J-01A-11R-A262-07,TCGA-S2-AA1A-01A-12R-A39D-07
Hybridization REF,,,,,,,,,,,,,,,,,,,,,
?|100130426,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
?|100133144,10.0113,7.1957,7.2453,11.3311,3.2254,4.0000,7.1084,3.4360,13.5406,9.4467,...,11.7148,3.6657,2.3298,16.3214,20.3514,15.5193,8.4195,42.9857,14.0861,20.5338
?|100134869,11.2820,12.4436,6.0184,7.5740,3.4942,13.7852,7.5810,12.1335,16.0273,10.4318,...,11.9573,9.7617,8.3410,7.5330,17.2393,22.9872,10.3226,81.1128,24.1914,8.9500
?|10357,49.5994,90.5117,49.5366,82.8303,72.5351,66.3658,109.1809,57.0596,108.4155,84.2496,...,163.5739,130.7486,101.7403,82.4231,100.4196,100.1867,74.7210,85.3715,61.1388,76.9265
?|10431,848.9397,924.0158,1140.6781,807.1729,562.0037,1342.6174,1148.3315,955.9141,844.5792,1397.9017,...,919.1686,403.4911,918.9533,995.6058,536.7602,873.9790,766.7448,677.7333,1137.3206,565.4335
?|136542,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
?|155060,345.2308,145.2025,51.7284,240.0221,274.2822,45.3020,175.6347,74.3578,190.2957,83.9315,...,291.5704,180.5975,541.1585,372.8814,240.4056,256.1260,370.2769,334.7453,170.8134,377.3925
?|26823,1.0472,1.6098,0.0000,0.4786,0.6109,0.3356,1.2773,0.5369,0.7582,0.8835,...,0.0000,0.0000,0.0000,0.6277,0.8742,1.1669,0.7209,0.0000,2.8708,0.6552
?|280660,0.0000,0.0000,0.0000,0.2393,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0000,1.8832,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000


## Step 3: Load and Clean the RNA-seq Matrix

We fix three problems in sequence:
1. Load full matrix skipping the junk header row
2. Filter out unnamed genes (those starting with `?`)
3. Clean gene names from `SYMBOL|entrez_id` → `SYMBOL`
4. Trim patient IDs to 12 chars and lowercase to match clinical data
5. Transpose so rows=patients, columns=genes

In [6]:
print("Loading full RNA-seq matrix — may take 15-20 seconds...")

rna_raw = pd.read_csv(
    RNASEQ_PATH,
    sep='\t',
    index_col=0,
    skiprows=[1],     # skip the 'normalized_count' junk row
    low_memory=False
)

print(f"Step 1 — Raw shape: {rna_raw.shape}  (genes × patients)")

# --- Fix 1: Filter out unnamed genes (starting with ?) ---
# rna_raw.index contains gene names like '?|100130426' and 'EGFR|1956'
# We keep only rows where the symbol before | is NOT '?'
named_mask = ~rna_raw.index.str.startswith('?')
rna_named = rna_raw[named_mask]
print(f"Step 2 — After removing ?| genes: {rna_named.shape}")

# --- Fix 2: Clean gene names from SYMBOL|entrez_id → SYMBOL ---
# 'EGFR|1956' → split on '|' → take first part → 'EGFR'
rna_named.index = rna_named.index.str.split('|').str[0]
print(f"Step 3 — Gene name examples after cleaning: {rna_named.index[:5].tolist()}")

# --- Fix 3: Trim patient IDs to 12 chars and lowercase ---
# 'TCGA-05-4244-01A-01R-1107-07' → 'TCGA-05-4244' → 'tcga-05-4244'
rna_named.columns = rna_named.columns.str[:12].str.lower()
print(f"Step 4 — Patient ID examples after cleaning: {rna_named.columns[:5].tolist()}")

# --- Transpose: rows=patients, columns=genes ---
rna = rna_named.T
print(f"Step 5 — After transpose: {rna.shape}  (patients × genes)")

Loading full RNA-seq matrix — may take 15-20 seconds...
Step 1 — Raw shape: (20531, 576)  (genes × patients)
Step 2 — After removing ?| genes: (20502, 576)
Step 3 — Gene name examples after cleaning: ['A1BG', 'A1CF', 'A2BP1', 'A2LD1', 'A2ML1']
Step 4 — Patient ID examples after cleaning: ['tcga-05-4244', 'tcga-05-4249', 'tcga-05-4250', 'tcga-05-4382', 'tcga-05-4384']
Step 5 — After transpose: (576, 20502)  (patients × genes)


## Step 4: Match Patients to Clinical Survival Labels

We keep only patients that appear in BOTH the RNA-seq matrix AND the 
clinical survival file. This is an inner join on patient IDs.

Patients in RNA-seq but not clinical → dropped (no survival label)
Patients in clinical but not RNA-seq → dropped (no expression data)
Only the intersection is usable for training.

In [7]:
# Load the cleaned clinical survival labels from Phase 3
clinical = pd.read_csv(
    'data/processed/clinical_survival.csv',
    index_col=0
)
print(f"Clinical patients:  {clinical.shape[0]}")
print(f"RNA-seq patients:   {rna.shape[0]}")

# Find the intersection of patient IDs
common_patients = clinical.index.intersection(rna.index)
print(f"\nPatients in both:   {len(common_patients)}")
print(f"Lost from clinical: {clinical.shape[0] - len(common_patients)}")
print(f"Lost from RNA-seq:  {rna.shape[0] - len(common_patients)}")

# Keep only common patients, in the same order
rna_matched = rna.loc[common_patients]
clinical_matched = clinical.loc[common_patients]

print(f"\nFinal matched shape:")
print(f"  RNA-seq:  {rna_matched.shape}  (patients × genes)")
print(f"  Clinical: {clinical_matched.shape}  (patients × columns)")

# Sanity check — indices must be identical
assert list(rna_matched.index) == list(clinical_matched.index), "Index mismatch!"
print("\nSanity check passed — patient order matches perfectly")

Clinical patients:  484
RNA-seq patients:   576

Patients in both:   478
Lost from clinical: 6
Lost from RNA-seq:  98

Final matched shape:
  RNA-seq:  (538, 20502)  (patients × genes)
  Clinical: (478, 10)  (patients × columns)


AssertionError: Index mismatch!

In [8]:
# Check for duplicate patient IDs in RNA-seq
print(f"Total RNA-seq patients: {rna.shape[0]}")
print(f"Unique RNA-seq patient IDs: {rna.index.nunique()}")
print(f"\nDuplicated IDs:")
duplicated = rna.index[rna.index.duplicated(keep=False)]
print(duplicated.value_counts().head(10))

Total RNA-seq patients: 576
Unique RNA-seq patient IDs: 516

Duplicated IDs:
tcga-38-4625    2
tcga-38-4626    2
tcga-50-5935    2
tcga-50-5936    2
tcga-50-5939    2
tcga-50-5946    2
tcga-50-6595    2
tcga-55-6968    2
tcga-55-6969    2
tcga-55-6970    2
Name: count, dtype: int64


## Step 5: Remove Duplicate Patients

Some TCGA patients had multiple tumour samples sequenced, resulting in 
duplicate rows. We keep the first occurrence per patient — standard 
practice in TCGA survival analysis.

In [9]:
# Keep first occurrence of each patient ID
rna_deduped = rna[~rna.index.duplicated(keep='first')]
print(f"Before deduplication: {rna.shape[0]} rows")
print(f"After deduplication:  {rna_deduped.shape[0]} rows")
print(f"Duplicates removed:   {rna.shape[0] - rna_deduped.shape[0]}")

# Now redo the matching with clean deduplicated matrix
common_patients = clinical.index.intersection(rna_deduped.index)
print(f"\nPatients in both after dedup: {len(common_patients)}")

rna_matched   = rna_deduped.loc[common_patients]
clinical_matched = clinical.loc[common_patients]

print(f"\nFinal matched shapes:")
print(f"  RNA-seq:  {rna_matched.shape}  (patients × genes)")
print(f"  Clinical: {clinical_matched.shape}  (patients × columns)")

# Sanity check
assert rna_matched.shape[0] == clinical_matched.shape[0], "Row count mismatch!"
assert list(rna_matched.index) == list(clinical_matched.index), "Index mismatch!"
print("\nSanity check passed — patient order matches perfectly")

Before deduplication: 576 rows
After deduplication:  516 rows
Duplicates removed:   60

Patients in both after dedup: 478

Final matched shapes:
  RNA-seq:  (478, 20502)  (patients × genes)
  Clinical: (478, 10)  (patients × columns)

Sanity check passed — patient order matches perfectly


## Step 6: Normalise Expression Values

Raw RSEM values range from 0 to thousands — too wide a range for neural networks.
We apply log2(x + 1) transformation which:
- Compresses large values into a manageable range (~0 to 15)
- Preserves zero values: log2(0 + 1) = 0
- Is the standard normalisation for RSEM data in published TCGA analyses

The +1 prevents log2(0) = negative infinity.

In [10]:
# Apply log2(x + 1) normalisation
# Every value x becomes log2(x + 1)
rna_log = np.log2(rna_matched + 1)

print("Normalisation complete")
print(f"\nBefore normalisation:")
print(f"  Min:  {rna_matched.values.min():.2f}")
print(f"  Max:  {rna_matched.values.max():.2f}")
print(f"  Mean: {rna_matched.values.mean():.2f}")

print(f"\nAfter log2(x+1) normalisation:")
print(f"  Min:  {rna_log.values.min():.2f}")
print(f"  Max:  {rna_log.values.max():.2f}")
print(f"  Mean: {rna_log.values.mean():.2f}")

Normalisation complete

Before normalisation:
  Min:  0.00
  Max:  1432694.16
  Mean: 977.83

After log2(x+1) normalisation:
  Min:  0.00
  Max:  20.45
  Mean: 6.54


In [11]:
rna_matched.shape

(478, 20502)

In [12]:
# Save full normalised matrix for CIBERSORT
# We need all genes — not the filtered 1,000 — to properly overlap with LM22
rna_log.to_csv('data/processed/expression_full_478.csv')

print(f"Saved full expression matrix: {rna_log.shape}")
print(f"Rows = patients, Columns = genes")
print(f"Location: data/processed/expression_full_478.csv")

Saved full expression matrix: (478, 20502)
Rows = patients, Columns = genes
Location: data/processed/expression_full_478.csv


## Step 7: Gene Selection — Two-Stage Filter

### Why not just use all 20,502 genes?
With only 478 patients, using all genes would cause severe overfitting.
We need to reduce dimensionality intelligently.

### Why not just variance?
High variance doesn't mean survival-relevant. A gene can vary across patients 
due to technical noise or cell type composition — not tumour biology.

### Our approach — Two stages:
**Stage 1 — Variance filter:** Keep top 5,000 most variable genes.
Cheap pre-filter that removes completely flat, uninformative genes.

**Stage 2 — Univariate Cox filter:** For each of the 5,000 genes, fit a 
single-gene Cox model and get a p-value. Keep top 1,000 genes most 
significantly associated with survival.

This directly connects gene selection to our prediction target.

In [13]:
# ── STAGE 1: Variance filter ──────────────────────────────────────────────
gene_variance = rna_log.var(axis=0)
top5000_genes = gene_variance.nlargest(5000).index
rna_var = rna_log[top5000_genes]

print(f"Stage 1 complete — Variance filter")
print(f"  Before: {rna_log.shape[1]:,} genes")
print(f"  After:  {rna_var.shape[1]:,} genes")

Stage 1 complete — Variance filter
  Before: 20,502 genes
  After:  5,000 genes


### Stage 2: Univariate Cox Filter

In [14]:
# ── STAGE 2: Univariate Cox filter ───────────────────────────────────────
from lifelines import CoxPHFitter
import warnings
warnings.filterwarnings('ignore')  # suppress convergence warnings for flat genes

# We need survival labels aligned to our 478 patients
survival_time  = clinical_matched['survival_time']
event          = clinical_matched['event']

# For each gene: fit a Cox model with just that gene as the feature
# Record the p-value — lower p-value = more significantly associated with survival
print("Running univariate Cox regression on 5,000 genes...")
print("This will take 3-5 minutes — each dot = 100 genes processed\n")

cox_pvals = {}

for i, gene in enumerate(rna_var.columns):
    if i % 100 == 0:
        print(f".", end='', flush=True)

    # Build a tiny dataframe for this one gene
    df = pd.DataFrame({
        'survival_time' : survival_time,
        'event'         : event,
        'gene'          : rna_var[gene]
    }).dropna()

    try:
        cph = CoxPHFitter()
        cph.fit(df, duration_col='survival_time', event_col='event')
        cox_pvals[gene] = cph.summary.loc['gene', 'p']
    except Exception:
        cox_pvals[gene] = 1.0  # if model fails, assign worst p-value

print(f"\n\nDone. {len(cox_pvals)} genes tested.")

# Convert to series and sort by p-value
cox_pvals = pd.Series(cox_pvals).sort_values()

print(f"\nTop 10 genes most associated with survival:")
print(cox_pvals.head(10))

Running univariate Cox regression on 5,000 genes...
This will take 3-5 minutes — each dot = 100 genes processed

..................................................

Done. 5000 genes tested.

Top 10 genes most associated with survival:
LINGO2       1.511155e-07
EPGN         2.260271e-06
DKK1         2.429903e-06
CD109        3.523769e-06
LOC441869    3.525884e-06
IRX5         1.438354e-05
DNAH2        2.627248e-05
INPP5J       3.129070e-05
FGF5         3.888469e-05
NTSR1        4.223933e-05
dtype: float64


### Select Top 1,000 Cox-significant Genes

In [17]:
# Keep top 1,000 genes by Cox p-value
TOP_N = 1000
top_genes = cox_pvals.head(TOP_N).index
rna_final = rna_var[top_genes]

print(f"Stage 2 complete — Univariate Cox filter")
print(f"  Before: 5,000 genes")
print(f"  After:  {rna_final.shape[1]:,} genes")
print(f"\nFinal expression matrix shape: {rna_final.shape}")
print(f"  Rows    = patients: {rna_final.shape[0]}")
print(f"  Columns = genes:    {rna_final.shape[1]}")
print(f"\nP-value cutoff of top 1,000 genes: {cox_pvals.iloc[999]:.6f}")
print(f"All selected genes have p < {cox_pvals.iloc[999]:.4f}")

Stage 2 complete — Univariate Cox filter
  Before: 5,000 genes
  After:  1,000 genes

Final expression matrix shape: (478, 1000)
  Rows    = patients: 478
  Columns = genes:    1000

P-value cutoff of top 1,000 genes: 0.072298
All selected genes have p < 0.0723


## Step 8: Save the Final Expression Matrix

We save two files:
- `expression_matrix.csv` — the 478 × 1,000 normalised, filtered expression matrix
- `selected_genes.txt` — the list of 1,000 selected gene names

The gene list is saved separately so Sonica can use it when computing 
dysregulation scores — she only needs to compute z-scores for these 1,000 genes,
not all 20,502.

In [18]:
# Save expression matrix
rna_final.to_csv('data/processed/expression_matrix.csv')
print(f"Saved expression matrix: {rna_final.shape}")
print(f"Location: data/processed/expression_matrix.csv")

# Save gene list separately — useful for Sonica's dysregulation pipeline
gene_list = pd.Series(rna_final.columns, name='gene')
gene_list.to_csv('data/processed/selected_genes.txt', index=False)
print(f"\nSaved gene list: {len(gene_list)} genes")
print(f"Location: data/processed/selected_genes.txt")

# Save Cox p-values for all 1,000 genes — useful for reporting
cox_pvals.head(TOP_N).to_csv('outputs/results/univariate_cox_pvalues.csv', header=True)
print(f"\nSaved Cox p-values")
print(f"Location: outputs/results/univariate_cox_pvalues.csv")

# Final summary
print(f"\n{'='*50}")
print(f"RNA-SEQ PROCESSING SUMMARY")
print(f"{'='*50}")
print(f"Raw genes loaded:        20,502")
print(f"After variance filter:    5,000")
print(f"After Cox filter:         1,000")
print(f"Patients matched:           478")
print(f"Normalisation:        log2(x+1)")
print(f"{'='*50}")
print(f"Ready for DeepSurv training")

Saved expression matrix: (478, 1000)
Location: data/processed/expression_matrix.csv

Saved gene list: 1000 genes
Location: data/processed/selected_genes.txt

Saved Cox p-values
Location: outputs/results/univariate_cox_pvalues.csv

RNA-SEQ PROCESSING SUMMARY
Raw genes loaded:        20,502
After variance filter:    5,000
After Cox filter:         1,000
Patients matched:           478
Normalisation:        log2(x+1)
Ready for DeepSurv training
